In [2]:
!pip install Sastrawi
!pip install emoji
!pip install sentence-transformers

In [3]:
import pandas as pd
import numpy as np
import json
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

import emoji

In [4]:
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Dataset_codingcamp_loker/data_untuk_modeling.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Preprocessing Text

In [5]:
df = pd.read_csv(file_path)
df.columns.tolist()

['Unnamed: 0',
 'Judul',
 'Industri',
 'Tipe',
 'Kategori',
 'Kota',
 'Provinsi',
 'Negara',
 'Gaji_Min',
 'Gaji_Max',
 'Skills',
 'Pengalaman_Bulan',
 'Pendidikan',
 'Deskripsi']

In [9]:
# mengecek kolom kosong
df.isnull().sum()

,0
Unnamed: 0,0
Judul,0
Industri,0
Tipe,0
Kategori,0
Kota,0
Provinsi,0
Negara,0
Gaji_Min,0
Gaji_Max,0


In [ ]:
df.head(3)

,Unnamed: 0,Judul,Industri,Tipe,Kategori,Kota,Provinsi,Negara,Gaji_Min,Gaji_Max,Skills,Pengalaman_Bulan,Pendidikan,Deskripsi
0,0,IT Business Analyst,Information Technology and Services,FULL_TIME,System Analyst,Jakarta Timur,DKI Jakarta,ID,Tidak ada,Tidak ada,"SQL, Business Analysis, Relational Databases, ...",36.0,bachelor degree,Job description \n \n Evaluating and documenti...
1,1,IT System Analyst,Information Technology and Services,FULL_TIME,System Analyst,Surabaya,Jawa Timur,ID,7000000.0,9000000.0,"Requirements Analysis, SQL, Business Analysis,...",12.0,bachelor degree,General Qualification : \n \n Man (23 - 35 yea...
2,2,IT Business Analyst - Insurance exp,Information Technology and Services,CONTRACTOR,Business Analyst,Jakarta Selatan,DKI Jakarta,ID,13000000.0,14500000.0,"SDLC, Life Insurance, Teamwork, Business Proce...",36.0,bachelor degree,Skill And Experience : \n - Understand the Bas...


In [6]:
df['text_gabungan'] = (
    "judul: " + df['Judul'].fillna('') + " " +
    "industri: " + df['Industri'].fillna('') + " " +
    "kategori: " + df['Kategori'].fillna('') + " " +
    "pendidikan: " + df['Pendidikan'].fillna('') + " " +
    "tipe: " + df['Tipe'].fillna('') + " " +
    "lokasi: " + (df['Kota'].fillna('') + " " + df['Provinsi'].fillna('')) + " " +
    "skills: " + df['Skills'].fillna('') + " " +
    "deskripsi: " + df['Deskripsi'].fillna('')
)

print(df['text_gabungan'][0])

judul: IT Business Analyst industri: Information Technology and Services kategori: System Analyst pendidikan: bachelor degree tipe: FULL_TIME lokasi: Jakarta Timur DKI Jakarta skills: SQL, Business Analysis, Relational Databases, System Analysis, Analytical Skills, SDLC deskripsi: Job description 
 
 Evaluating and documenting business processes, anticipating requirements, uncovering areas for improvement, and developing and implementing solutions. 
 Conducting meetings and presentations to share ideas and findings. 
 Performing requirements analysis. 
 Working closely with clients, technical/programer, and project manager 
 Create Business Requirement documents and Functional Specification Document 
 
 
 Qualification 
 Minimum Bachelor’s Degree in Business ot  IT from reputable university 
 
 Minimum 2 (two) year experience as Business Analyst or Functional Analyst 
 Having good knowledge of Software Development Life Cycle  (SDLC) 
 Excellent documentation skills. 
 Excellent analyti

##### Cleaning

In [7]:

def basic_clean(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)                 # HTML
    text = re.sub(r'[^a-z0-9\s:/_\-\+\.]', ' ', text) # keep tanda penting
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_gabungan'] = df['text_gabungan'].apply(basic_clean)
print(df['text_gabungan'][0])

judul: it business analyst industri: information technology and services kategori: system analyst pendidikan: bachelor degree tipe: full_time lokasi: jakarta timur dki jakarta skills: sql business analysis relational databases system analysis analytical skills sdlc deskripsi: job description evaluating and documenting business processes anticipating requirements uncovering areas for improvement and developing and implementing solutions. conducting meetings and presentations to share ideas and findings. performing requirements analysis. working closely with clients technical/programer and project manager create business requirement documents and functional specification document qualification minimum bachelor s degree in business ot it from reputable university minimum 2 two year experience as business analyst or functional analyst having good knowledge of software development life cycle sdlc excellent documentation skills. excellent analytical judgment and problem solving skills high q

##### Case Folding

In [8]:
def case_folding(text):
    text = text.lower()
    return text

df["text_gabungan"] = df["text_gabungan"].apply(case_folding)
print(df["text_gabungan"][1])

judul: it system analyst industri: information technology and services kategori: system analyst pendidikan: bachelor degree tipe: full_time lokasi: surabaya jawa timur skills: requirements analysis sql business analysis sdlc stakeholder management cloud platform system teamwork system analysis deskripsi: general qualification : man 23 - 35 years old education min. s1 information technology/information system/computer system minimum 1-3 years of experience in it coordination business partnering or similar roles. strong understanding of it service management project management and technology roadmaps. strong communication collaboration and stakeholder management skills. fluent in bahasa indonesia and english written and spoken . will to work located in surabaya specific qualification : sql query/tuning basic programming language laravel basic understand client-server and networking concept basic it service management knowledge basic familiar with report development ssrs powerbi will be a

##### Normalization

In [ ]:
df_normalisasi = pd.read_csv("colloquial-indonesian-lexicon-normalisasi.csv")
df_normalisasi = df_normalisasi[["slang", "formal"]]
df_normalisasi.head()

,slang,formal
0,woww,wow
1,aminn,amin
2,met,selamat
3,netaas,menetas
4,keberpa,keberapa


In [ ]:
# normalization_dict = dict(zip(df_normalisasi['slang'], df_normalisasi['formal']))
# normalization_dict.pop('it')

# def normalization(text):
#     words = text.split()
#     words = [normalization_dict.get(word, word) for word in words]
#     return ' '.join(words)

# df['text_gabungan'] = df["text_gabungan"].apply(normalization)
# print(df['text_gabungan'][1])

judul: itu system analyst industri: information technology and services kategori: system analyst pendidikan: bachelor degree tipe: full_time lokasi: surabaya jawa timur skills: requirements analysis sql business analysis sdlc stakeholder management cloud platform system teamwork system analysis deskripsi: general qualification : man 23 - 35 years old education min. s1 information technology/information system/computer system minimum 1-3 years of experience ini itu coordination business partnering orang similar roles. strong understanding of itu service management project management and technology roadmaps. strong communication collaboration and stakeholder management skills. fluent ini bahasa indonesia and english written and spoken . will tapi work located ini surabaya specific qualification : sql query/tuning basic programming language laravel basic understand client-server and networking concept basic itu service management knowledge basic familiar with report development ssrs power

##### Stopword Removal

In [ ]:
# tidak relevan untuk kasus INDO ENGLISH, karena banyak kata yang sudah baku dan tidak ada slang yang perlu dinormalisasi dan Stopword

#### Membangun Modeling dengan Pseudo-CV
Input Text  
   ↓  
BERT Encoder  
   ↓  
Pooling (CLS / mean pooling)  
   ↓  
Dense (256) + ReLU  
   ↓  
Dense (128)  ← embedding akhir  
   ↓  
L2 Normalization  

#### Membangun Data Pseudo-CV

In [9]:
df['pseudo_cv'] = (
    "judul: " + df['Judul'].fillna('') + " " +
    "industri: " + df['Industri'].fillna('') + " " +
    "kategori: " + df['Kategori'].fillna('') + " " +
    "pendidikan: " + df['Pendidikan'].fillna('') + " " +
    "tipe: " + df['Tipe'].fillna('') + " " +
    "skills: " + df['Skills'].fillna('') + " "
)
print(df['pseudo_cv'][0])

judul: IT Business Analyst industri: Information Technology and Services kategori: System Analyst pendidikan: bachelor degree tipe: FULL_TIME skills: SQL, Business Analysis, Relational Databases, System Analysis, Analytical Skills, SDLC 


In [1]:
# from sentence_transformers import SentenceTransformer, InputExample
# from sentence_transformers.sentence_transformer import losses
# from torch.utils.data import DataLoader

# model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
# train_examples = [
#     InputExample(texts=[df['pseudo_cv'], df['text_gabungan']])
#     # for pseudo_cv, job_full in pairs
#     for df['pseudo_cv'], df['text_gabungan'] in zip(df['pseudo_cv'], df['text_gabungan'])
# ]

# train_loader = DataLoader(train_examples, batch_size=16, shuffle=True)
# loss_fn = losses.MultipleNegativesRankingLoss(model)

# model.fit(
#     train_objectives=[(train_loader, loss_fn)],
#     epochs=1
# )

#### Inferens

In [ ]:
# cv_emb = model.encode(real_cv_text)
# job_embs = model.encode(job_list)

# scores = cosine_similarity([cv_emb], job_embs)

In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Dot, Flatten
from tensorflow.keras.models import Model

In [11]:
cv_texts = []
job_texts = []
labels = []

# Data Positif (Cocok = 1) -> pseudo_cv dipasangkan dengan job aslinya
for i in range(len(df)):
    cv_texts.append(df['pseudo_cv'].iloc[i])
    job_texts.append(df['text_gabungan'].iloc[i])
    labels.append(1)

# Data Negatif (Tidak Cocok = 0) -> pseudo_cv dipasangkan dengan job random
# Ini penting agar model LSTM tidak sekadar menebak "1" terus menerus
for i in range(len(df)):
    random_idx = np.random.choice([x for x in range(len(df)) if x != i])
    cv_texts.append(df['pseudo_cv'].iloc[i])
    job_texts.append(df['text_gabungan'].iloc[random_idx])
    labels.append(0)

# Convert ke numpy array
labels = np.array(labels)

In [12]:
MAX_VOCAB = 10000
# masih perlu diteliti lebih lanjut
MAX_LEN = 150

In [13]:
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
# Fit tokenizer ke semua teks (CV + Job)
tokenizer.fit_on_texts(cv_texts + job_texts)

# Convert teks ke sequence angka dan lakukan padding
cv_seq = pad_sequences(tokenizer.texts_to_sequences(cv_texts), maxlen=MAX_LEN, padding='post')
job_seq = pad_sequences(tokenizer.texts_to_sequences(job_texts), maxlen=MAX_LEN, padding='post')

In [14]:
# 3. MEMBANGUN ARSITEKTUR CUSTOM LAYER (Siamese BiLSTM)
embedding_dim = 64

In [15]:
# Input Layers
cv_input = Input(shape=(MAX_LEN,), name="cv_input")
job_input = Input(shape=(MAX_LEN,), name="job_input")

# Embedding Layer (Dibagi / Share weights untuk kedua input)
shared_embedding = Embedding(input_dim=MAX_VOCAB, output_dim=embedding_dim, name="shared_embedding")

cv_emb = shared_embedding(cv_input)
job_emb = shared_embedding(job_input)

# BiLSTM Layer (Dibagi / Share weights untuk kedua input)
shared_bilstm = Bidirectional(LSTM(64, return_sequences=False), name="shared_bilstm")

cv_vector = shared_bilstm(cv_emb)
job_vector = shared_bilstm(job_emb)

In [16]:
# Menghitung kemiripan (Dot Product sering dipakai untuk mengukur similarity di neural network)
similarity = Dot(axes=1, normalize=True, name="cosine_similarity")([cv_vector, job_vector])

# Gabungkan menjadi satu model
model = Model(inputs=[cv_input, job_input], outputs=similarity)

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cv_input            │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_input           │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_embedding    │ (None, 150, 64)   │    640,000 │ cv_input[0][0],   │
│ (Embedding)         │                   │            │ job_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bilstm       │ (None, 128)       │     66,048 │ shared_embedding… │
│ (Bidirectional)     │                   │            │ shared_embedding… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cosine_similarity   │ (None, 1)         │          0 │ shared_bilstm[0]… │
│ (Dot)               │                   │            │ shared_bilstm[1]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 706,048 (2.69 MB)

 Trainable params: 706,048 (2.69 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
from sklearn.model_selection import train_test_split

X_cv_train, X_cv_test, X_job_train, X_job_test, y_train, y_test = train_test_split(
    cv_seq,
    job_seq,
    labels,
    test_size=0.15,
    random_state=42
)

print(f"Total data Train (termasuk Validation nanti): {len(y_train)}")
print(f"Total data Test murni: {len(y_test)}")

Total data Train (termasuk Validation nanti): 4693
Total data Test murni: 829


In [21]:
# 4. TRAINING MODEL
# Karena target kita asal jalan dulu, 5-10 epoch sudah cukup
model.fit(
    x=[X_cv_train, X_job_train],
    y=labels,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - loss: 0.2230 - mae: 0.3574 - val_loss: 0.5234 - val_mae: 0.7184
Epoch 2/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - loss: 0.1973 - mae: 0.3793 - val_loss: 0.6029 - val_mae: 0.7749
Epoch 3/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.1920 - mae: 0.3782 - val_loss: 0.6284 - val_mae: 0.7903
Epoch 4/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.1843 - mae: 0.3668 - val_loss: 0.6002 - val_mae: 0.7694
Epoch 5/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - loss: 0.1702 - mae: 0.3410 - val_loss: 0.5065 - val_mae: 0.6994


In [23]:
import pickle

# Tentukan path penyimpanan di Google Drive kamu
# Menggunakan folder yang sama dengan tempat dataset kamu berada
save_path = '/content/drive/MyDrive/Dataset_codingcamp_loker/'

# 1. Menyimpan Model Keras
# Menggunakan format .keras yang merupakan standar terbaru dan terefisien dari TensorFlow
model.save(save_path + 'model_rekomendasi_loker.keras')

# 2. Menyimpan Tokenizer menggunakan Pickle
with open(save_path + 'tokenizer_loker.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Model dan Tokenizer berhasil disimpan dengan aman di Google Drive!")

Model dan Tokenizer berhasil disimpan dengan aman di Google Drive!


In [22]:
print("\n=== Evaluasi pada Data Test Murni ===")
eval_results = model.evaluate(x=[X_cv_test, X_job_test], y=y_test)

print(f"Loss pada Data Test: {eval_results[0]:.4f}")
print(f"MAE pada Data Test: {eval_results[1]:.4f}")


=== Evaluasi pada Data Test Murni ===
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.2913 - mae: 0.4802
Loss pada Data Test: 0.2913
MAE pada Data Test: 0.4802
